# Notebook 14 — Performance Optimization

## Learning objectives
Use vectorization, choose efficient dtypes, reduce memory, and write maintainable Pandas code.

**How to use this notebook:** Read each Markdown cell, run the following Python cell, compare your output with the explanation, and then try changing the sample values. The examples use small, fictional datasets so they can run locally without downloading data.

## Setup
NumPy and Pandas are common Python libraries for numerical computation and tabular data analysis. If they are not installed, run `pip install numpy pandas openpyxl pyarrow` in your terminal or notebook environment.

In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.3.5
Pandas version: 2.2.3


## Sample dataset

In [2]:
df = pd.DataFrame({
    'store': ['A', 'B', 'A', 'C', 'B', 'A'],
    'units': [10, 20, 15, 12, 18, 9],
    'price': [2.5, 3.0, 2.5, 4.0, 3.0, 2.5]
})
print(df)

  store  units  price
0     A     10    2.5
1     B     20    3.0
2     A     15    2.5
3     C     12    4.0
4     B     18    3.0
5     A      9    2.5


## Vectorization and apply() vs loops

### 1. Concept explanation
Vectorization performs operations over whole arrays/columns; loops and row-wise `apply` often add Python-level overhead.

### 2. Real-world analogy
A machine processes a batch at once instead of handling each item individually.

### 3. Business example
Calculate revenue for a large order table.

### 4. AI/ML example
Prepare numeric features efficiently.

In [3]:
df['revenue_vectorized'] = df['units'] * df['price']
print(df[['units', 'price', 'revenue_vectorized']])
# Row-wise apply is possible but usually unnecessary for this multiplication:
# df['revenue_apply'] = df.apply(lambda row: row['units'] * row['price'], axis=1)

   units  price  revenue_vectorized
0     10    2.5                25.0
1     20    3.0                60.0
2     15    2.5                37.5
3     12    4.0                48.0
4     18    3.0                54.0
5      9    2.5                22.5


### 5. Output explanation
Each revenue is units multiplied by price. The vectorized expression is clear and avoids a row-wise Python function.

### 6. Best practices
Prefer vectorized Series/NumPy operations when possible; benchmark real workloads.

### 7. Common mistakes
Assuming every apply is slow is too broad; performance depends on the operation.

### 8. Advantages
Often faster and simpler.

### 9. Limitations
Some complex logic has no simple vectorized equivalent.

## Memory optimization and category dtype

### 1. Concept explanation
Use suitable numeric dtypes and categorical dtype for repeated labels to reduce memory use.

### 2. Real-world analogy
Store repeated labels as a shared codebook rather than full text each time.

### 3. Business example
Optimize large product, region, or status columns.

### 4. AI/ML example
Reduce memory pressure when loading large training tables.

In [4]:
before = df.memory_usage(deep=True).sum()
df['store'] = df['store'].astype('category')
after = df.memory_usage(deep=True).sum()
print('Before bytes:', before)
print('After bytes:', after)
print(df.dtypes)

Before bytes: 576
After bytes: 540
store                 category
units                    int64
price                  float64
revenue_vectorized     float64
dtype: object


### 5. Output explanation
Memory usage is reported before and after converting store to category; the result depends on dataset size and cardinality.

### 6. Best practices
Categoricals are most useful for repeated values with limited unique categories.

### 7. Common mistakes
Converting a nearly unique column to category may not save memory.

### 8. Advantages
Can reduce memory significantly.

### 9. Limitations
Categorical conversion adds little benefit for high-cardinality columns.

## Efficient Pandas practices

### 1. Concept explanation
Load only needed columns, set appropriate types, avoid repeated concatenation, use built-in aggregations, and inspect memory.

### 2. Real-world analogy
Carry only the tools needed for a task rather than a whole toolbox.

### 3. Business example
Read selected columns from large transaction exports.

### 4. AI/ML example
Build robust data pipelines that scale better.

In [5]:
# Example: load only needed CSV columns
# subset = pd.read_csv('transactions.csv', usecols=['date', 'amount', 'merchant'])
# Efficiently combine many small frames once:
parts = [pd.DataFrame({'value': [i, i + 1]}) for i in range(0, 6, 2)]
combined = pd.concat(parts, ignore_index=True)
print(combined)
print('Memory bytes:', combined.memory_usage(deep=True).sum())

   value
0      0
1      1
2      2
3      3
4      4
5      5
Memory bytes: 180


### 5. Output explanation
The frames are concatenated once into one table and memory use is estimated.

### 6. Best practices
Profile before optimizing and keep code readable.

### 7. Common mistakes
Premature optimization can make code hard to maintain without measurable benefit.

### 8. Advantages
Improves scalability and clarity.

### 9. Limitations
Performance depends on data shape, hardware, and operations.